# Datamine CSOWOPT Process Example

This notebook demonstrates how to configure and run the **`csowopt`** process wrapper in `dmstudio`.

## Process Description

## CSOWOPT

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process identifies ore and waste blocks in a conditionally simulated model by minimizing the cost of incorrectly classifying ore as waste or waste as ore.

Reserves are calculated for a range of costs and prices for each cutoff, allowing the sensitivity of the reserves to changes in these parameters to be quantified.

The **CSOWOPT** process carries out the following functions:

* Inputs a conditionally simulated block model statistics file as created by [CSMODEL](<csmodel.md>).
* Inputs a range of processing costs and metal prices as defined by a minimum value, maximum value and increment. A single cost or price is selected by setting the minimum equal to the maximum.
* Calculates whether each block is ore or waste by minimizing the cost of misclassifying ore as waste or waste as ore for each combination of cost, price and cutoff.
* Creates a reserves table of tonnes and grade classified by cost, price and cutoff.
* Creates an output model identifying ore and waste blocks for each cutoff. This model is only created for the minimum cost and minimum price values

### Input Files:

* **statmod** (Block Model):
  Conditionally simulated block model statistics file. This file will have been created
  as the output STATMOD file by CSMODEL, and must include the fields PAx, GAx and GBx for
  at least one value of cutoff grade x.
  Required=Yes

### Output Files:

* **reserves** (Table):
  Output reserves file containing total tonnes and grade for blocks calculated as ore,
  classified by processing cost, metal price and cutoff.
  Required=Yes

* **oremod** (Block Model):
  Output block model. This file includes flag field OWx which equals 1 if the block is
  ore and 0 if it is waste, for each cutoff x, for processing cost COSTMIN and metal price

## PRICEMIN .

  Required=No

* **plot** (Plot):
  Template name for output plot file(s) showing tonnes, grade and/or metal above cutoff
  (Y axis) against cutoff (X axis). The PLOT file template name should be a maximum of 23
  characters. A single character is added to this name to create the actual file name, as

* **follows** (G - Grade T - Tonnes M - Metal The parameters GPLOT , TPLOT and MPLOT define):
  which plots to create. A minimum of 2 cutoffs must have been defined in order for the
  PLOT file(s) to be created.
  Required=No

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('csowopt')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute csowopt
print("Running csowopt...")
dm_cmd.csowopt(
    statmod_i='t_input_file',  # required
    reserves_o=['t_csowopt_out'],  # required
    # oremod_o='t_csowopt_out',  # optional
    # plot_o='t_csowopt_out',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("csowopt execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_csowopt_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")